## 1. Setup and Artefact Load.
Prepare the training environment and load the unified dataset artefacts created during `01_data_eda.ipynb`.
#### Rationale:
- Establish a **reproducible** experimental context (fixed random seed and deterministic cuDNN - NVIDIA CUDA Deep Neural Network library).
- Resolve canonical project paths and verify existence of required directories (`data`, `experiments/pools/`, etc.).
- Load the **pooled table** (`source_pool_rel.csv`) and the **preprocessing configuration** (`preprocessing_config.json`) that define the 3-class problem (`Normal`, `Pneumonia`, `TB`).
- Create the output directories for checkpoints and metrics to keep training runs auditable.
- Print a concise summary confirming *GPU availability*, *ImageNet normalisation*, and *reproducibility* settings.

In [ ]:
from pathlib import Path
import os, json, random, numpy as np, pandas as pd, torch

# 1. Reproducibility.
#   Use a fixed seed across all random libraries (Python, NumPy, and PyTorch)
# to make training results numerically repeatable on this workstation and future reruns.
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Disable non-deterministic kernels.
    torch.backends.cudnn.benchmark = False # Prevent cuDNN autotune drift.

SEED = 42
set_seed(SEED)

# 2. Resolve canonical project paths.
#   This notebook lives in notebooks/, therefore ".." points to the repository root.
PROJECT_ROOT    = Path("..").resolve()
CHEXPERT_PATH   = PROJECT_ROOT / "data" / "chexpert"
TBX11K_PATH     = PROJECT_ROOT / "data" / "tbx11k"
POOLS_DIR       = PROJECT_ROOT / "experiments" / "pools"
CHECKPOINTS_DIR = PROJECT_ROOT / "experiments" / "checkpoints"
RESULTS_DIR     = PROJECT_ROOT / "experiments" / "results"

# 3. Load persisted artefacts from 01_data_EDA.
#   The pooled CSV contains relative paths, class labels, and source tags.
# The JSON holds preprocessing constants (IMG_SIZE, ImageNet mean/std, label map, transform mode).
pool_csv = POOLS_DIR / "source_pool_rel.csv"
cfg_path = POOLS_DIR / "preprocessing_config.json"

if not pool_csv.exists() or not cfg_path.exists():
    raise FileNotFoundError("Missing pooled artefacts. Please run 01_data_eda.ipynb first "
                            "to generate experiments/pools/source_pool_rel.csv and preprocessing_config.json."
    )
source_pool_rel = pd.read_csv(pool_csv)
cfg = json.loads(cfg_path.read_text())

CLASS_TO_INDEX = {k: int(v) for k, v in cfg["class_to_index"].items()}
INDEX_TO_CLASS = {v: k for k, v in CLASS_TO_INDEX.items()}
IMG_SIZE       = int(cfg["img_size"])
IMAGENET_MEAN  = cfg["imagenet_mean"]
IMAGENET_STD   = cfg["imagenet_std"]
TRANSFORM_MODE = cfg.get("transform_mode", "center-crop")

# 5. Device and precision information.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Automatic Mixed Precision (AMP). Safely enable it for faster training.
USE_AMP = bool(device.type == "cuda") # Enable mixed precision if a GPU is present.

# 6. Compact readiness log.
print("03_model_training - Environment")
print(f'- Device: {device.type.upper()} | PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
if device.type == "cuda":
    print(f'- GPU: {torch.cuda.get_device_name(0)} | cuDNN: {torch.backends.cudnn.version()}')

print("\nInputs (from 01/02):")
print(f'- Pool rows: {len(source_pool_rel):,}')
print(f'- Class map: {CLASS_TO_INDEX}')
print(f'- Image size: {IMG_SIZE} | Transform mode: {TRANSFORM_MODE}')
print(f'- ImageNet mean/std: {IMAGENET_MEAN} / {IMAGENET_STD}')

print("\nOutputs:")
print(f'- Checkpoints: {CHECKPOINTS_DIR.relative_to(PROJECT_ROOT)}')
print(f'- Metrics: {RESULTS_DIR.relative_to(PROJECT_ROOT)}')

print("\nReproducibility:")
print(f'- Seed: {SEED} | Deterministic cuDNN: {torch.backends.cudnn.deterministic}')
print(f'- Precision: {"Mixed (AMP)" if USE_AMP else "FP32"}')

## 2. Module Import Path Setup.
Ensure that the project's internal modules stored under '/src' (such as `/src/dataset.py`,`/src/transforms.py`, and `/src/loaders.py`) are discoverable when the notebook is executed from within the `/notebooks` directory.
By explicitely adding the parent directory to `sys.path`, we make sure that Python recongnises the custom package structure regardless of runtime environment (local workstation, remote server, or container).
#### Rationale:
- Avoids import errors when running notebooks from nested directories by dynamically adding the project root to the Python path.
- Guarantees **modular reproducibility** and **environmental portability** across local and cloud-based executions.
- Aligns the notebook workflow with the structure used for later **containerised and Kubernetes deployments**.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root (one level up from 'notebooks/') is visible to Python
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to sys.path:", PROJECT_ROOT)

## 3. Shared Preprocessing Transforms.
Construct the shared preprocessing pipelines for both **training** and **evaluation** datasets using the parameters persisted during *01_data_eda.ipynb*.
These transforms include deterministic resizing, cropping, and ImageNet-compatible normalisation, ensuring that both CheXpert and TBX11K images are processed identically.
#### Rationale:
- Enforces **standarised input statistics** across datasets for fair comparison between custom and pretrained models.
- Aligns CXR input space with the **ImageNet pretraining domain**, enabling effective transfer learning.
- Promotes **reproducibility** and **consistency** in all subsequent experiments by applying identical preprocessing logic.

In [ ]:
# Shared preprocessing transforms (Train / Eval)
# Build shared preprocessing transforms using parameters from preprocessing_config.json.
# These transforms ensure consistent resizing, cropping, and ImageNet normalisation
# across both datasets (CheXpert and TBX11K).

from src.transforms import build_shared_transforms

train_transforms_shared, eval_transforms_shared = build_shared_transforms(
    img_size=IMG_SIZE,
    imagenet_mean=IMAGENET_MEAN,
    imagenet_std=IMAGENET_STD,
)

print(f'Transforms built successfully: {IMG_SIZE}×{IMG_SIZE} (ImageNet-normalised)')

## 4. Stratified Train/Validation DataLoaders.
Build the final **train** and **validation** DataLoaders by splitting the unified dataset into 80/20 **stratified** partition.
This ensures that all three classes, `Pneumonia`, `TB`, and `Normal`, maintain proportional representation in both subsets.
Attach the appropriate transform pipeline to each split and configure **DataLoaders** optimised for GPU throughput.
#### Rationale:
- Preserves **class balance** across splits, preventing training bias and enabling robust evaluation.
- Ensures that **data augmentation** is applied only to the **training** subset while **evaluation** remains deterministic.
- Uses **parallel workers** and **pinned memory* to maximise GPU utilisation and minimise I/O latency.
- Facilitates **reproducible, high-throughput training** under fixed seeds and controlled environment conditions.

In [ ]:
from src.loaders import build_dataloaders

train_loader, val_loader = build_dataloaders(
    source_pool_rel=source_pool_rel,
    class_to_index=CLASS_TO_INDEX,
    chexpert_root=CHEXPERT_PATH,
    tbx_root=TBX11K_PATH,
    img_size=IMG_SIZE,
    train_transforms=train_transforms_shared,
    eval_transforms=eval_transforms_shared,
    seed=SEED,
    batch_size=64,
    num_workers=4,
)

## 5. Model Initialisation and Training Configuration.
Define and initialise the neural network architecture, loss function, and optimisation strategy used for model training and evaluation.
This section ensures that both **custom** and **pretrained** models follow a unified configuration.
#### Rationale:
- Maintain **consistency** across all model types (**custom CNN** and **transfer-learning backbones**).
- Enable **reproducible training** by fixing seeds and using identical optimiser and scheduler settings.
- Support **fair comparison** of model performance under controlled experimental conditions.

In [ ]:
# Model Initialisation and Training Configuration.
# Define the selected model architecture, loss function, optimiser,and scheduler.
# This configuration ensures consistent setup for both custom and transfer-learning backbones
# (MobileNetV@, Efficient-B0, ResNet-50) as well as the baseline CNN.

from torchvision import models
import torch.nn as nn
import torch.optim as optim

# 1. Selection model architecture.
#   Options: 'custom', 'mobilenet_v2', 'efficientnet_b0', 'resnet5'
MODEL_NAME = "mobilenet_v2" # Example: can be changed per experiment.

if MODEL_NAME == "mobilenet_v2":
    model = models.mobilenet_v2(weights="IMAGENET1K_V1")
    model.classifier[1] = nn.Linear(model.last_channel, len(CLASS_TO_INDEX))
elif MODEL_NAME == "efficientnet_b0":
    model = models.efficientnet_b0(weights="IMAGENET1K_V1")
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASS_TO_INDEX))
elif MODEL_NAME == "resnet50":
    model = models.resnet50(weights="IMAGENET1K_V1")
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_TO_INDEX))
elif MODEL_NAME == "vitorio":
    # Example baseline CNN (placeholder, will be defined later if needed).
    # The name of the custom model is Vitorio:v_1.0 in memory of Dr. Vitorio Corradi,
    # researcher's medical-doctor mentor in respiratory diseases during his medical training
    # at Universidad de Valparaíso and Hospital San Camilo, San Felipe, Chile, 2006-2012.
    model = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * (IMG_SIZE // 4) * (IMG_SIZE // 4), len(CLASS_TO_INDEX)),
    )
else:
    raise ValueError(f'Unknown model: {MODEL_NAME}')

# Move model to GPU if available.
model = model.to(device)

# 2. Define loss function.
#   In essence, the loss function is the guide that tells the network how well the model is doing.
#   Cross entropy loss is chosen due to its good performance in classification tasks.
#   Cross entropy loss, cross-entropy, or logarithmic loss, is the most popular loss function
#   for classification problems because it is very effective at guiding the model to output
#   high probabilities for the correct class and low probabilities for wrong ones.
#   In plain terms cross entropy loss penalises the model for being confident about the wrong
#   answer, or for being unsure about the right answer. 
#   A model gets the lowest loss if it assigns a propability of 1 to the correct class 
#   and 0 to the others.
#   A model gets the highest loss if it assigns a probability of 0 to the right class or gives
#   higher probabilities to the incorrect ones.
criterion = nn.CrossEntropyLoss()

# 3. Define optimiser.
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

# 4. Define learning rate scheduler.
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# 5. Print readiness summary.
print(f'Model initialised: {MODEL_NAME}')
print(f'- Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'- Device: {device}')
print(f'- Optimiser: {optimizer}')
print(f'- Scheduler: {scheduler}')    

## 6. Debug Confirmation (CUDA and Device Sanity Check).
Verify that the GPU environment is correctly initialised before training.
This diagnostic step confirms that PyTorch can access the NVIDIA RTX 5090 device, initialise CUDA contexts, and allocate tensors for computation.
# Rationale:
- Ensures **CUDA and cuDNN** are properly loaded and functional in the active environment.
- Confirms **hardware acceleration** is available for model training and evaluation.
- Prevents silent fallbacks to CPU execution, maintaining **reproducibility and performance consistency**.

In [ ]:
# GPU diagnostic and model sanity check.
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("Memory allocated:", torch.cuda.memory_allocated() / 1e9, "GB")
    print("Memory reserved:", torch.cuda.memory_reserved() / 1e9, "GB")

# Quick sanity check: forward pass of a dummy batch.
x = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(device) # Random batch of 4 images.
with torch.no_grad():
    out = model(x)
print("Output shape:", out.shape)

## 7. Training Loop and Metrics Tracking.
Train and validate the selected model over multiple epochs while monitoring loss and accuracy. This section implements a reproducible supervised training loop compatible custom and transfer-learning architectures.
#### Rationale:
- Executes **iterative optimatisation** of network weights over mini-batches.
- Monitors **training vs validation performance** to detect overfitting.
- Saves best checkpoints and metrics for later **statistical testing** and reproducibility.

In [ ]:
# 1. Configuration parameters.

from torch.cuda.amp import autoscast, GradScaler
from tqdm import tqdm
import time, json

# 1. Configuration parameters.
NUM_EPOCHS = 10 # Number of complete passes through the dataset.
best_val_acc = 0.0 # Track best validation accuracy achieved so far.
# USE: Automatic Mixed Precision package - torch.amp
# Some ops, like linear layers and convolutions, are much fuster in 'lower precission_fp'. Other ops, like reductions,
# often require the dynamic range of 'float32'. Mixed precision tries to match each op to its appropriate datatype.
scaler = GradScaler(enabled=USE_AMP) # Handle safe scaling when mixed automatic main precision (AMP) is enabled.

# Initialise a metrics dictionary to record training dynamics across epochs.
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [], 
    "train_acc": [], # Training accuracy.
    "val_acc": [], # Validation accuracy.
{

# 2. Helper functions.
def accuracy(preds, labels):  # preds(logits): raw output numbers that a machine learning models produces before 
                              # converting them into probabilities. They are like confidence scores, not yet
                              # squished into a range between 0 and 1.
    """
    Compute classification accuracy as the mean proportion of correct predictions.
    Args:
        preds  (Tensor): Model output logits for each class.
        labels (Tensor): Ground-truth class indices.
    Returns:
        float: Accuracy in range [0, 1].
    """
    return ([preds.argmax(dim=1) == labels.float().mean().item()

def run_epoch(loader, model, criterion, optimizer=None, train=False):
    """
    Execute one full pass (epoch) over a DataLoader, performing training or evaluation.

    Args: 
        loader (DataLoader): The dataset iterator (train or validation).
        model (nn.Module): Current neural network instance.
        criterion: Loss function used to quantify prediction error.
        optimizer (optional): Optimiser for gradient descent (only if training).
                              Gradient is the vector of partial derivatives of the
                              loss with respect to each paramenter. It essencially
                              tells which direction is downhill in the slope of the
                              loss function in the higher dimensional space of the
                              networks's weights.
                              By computing the gradient of the loss it is possible
                              to adjust each weight in order to reduce the error a
                              little bit.
                              Back propagation is the algorithm that efficiently
                              computes these gradients for all the weights in a
                              multilayer neural network.
                              After the network does a forward pass on some
                              training data, and the loss function produces a 
                              loss value or the error, back propagation works 
                              backwards from the output layer to the input.
                              This is essentianlly the chain rule of calculus applied
                              to the network's composed functions breaking down the
                              losses derivative layer by layer.
                              The result of back propagation is the gradient of the loss
                              with respect to every weight and bias in the network.
                              calculating how much each weight contributed to
                              the error.
        train (bool): true enables gradient updates: False disables them.

    Returns:
        tuple: (average_loss, average_accuracy)
    """